# Model A — Cross-Entropy baseline

5-fold GroupKFold, no augmentation. Kaggle public LB: 0.773.


In [1]:
from google.colab import drive
drive.mount("/content/drive")
%run colab_init.py

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn matplotlib seaborn kaggle

from birdclef.colab import colab_prepare_training

colab_prepare_training(stage_tta=False)



Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.7 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [ ]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [ ]:
df, NUM_CLASSES, _ = prepare_baseline_data()
print(f"Samples: {len(df)}, classes: {NUM_CLASSES}")
print(df["fold"].value_counts().sort_index())
model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)


Samples: 35549, classes: 234
fold
0    7110
1    7110
2    7110
3    7110
4    7109
Name: count, dtype: int64


In [ ]:
SAVE_DIR = MODELS_DIR / "ce_baseline_10ep"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 10
baseline_fold_scores = []

for TRAIN_FOLD in range(5):
    best_auc = 0.0

    # Split data by fold
    train_df = df[df["fold"] != TRAIN_FOLD].reset_index(drop=True)
    valid_df = df[df["fold"] == TRAIN_FOLD].reset_index(drop=True)
    train_ds = BirdDataset(train_df, is_tta=False)
    valid_ds = BirdDataset(valid_df, is_tta=False)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=0)
    fold_train_losses = []
    fold_val_losses = []
    fold_val_aucs = []

    print(f"TRAINING FOLD {TRAIN_FOLD}")
    print(f"Training on {len(train_df)} samples, validating on {len(valid_df)} samples")

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        avg_val_loss = val_loss / len(valid_loader)
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)

        # One-hot encode labels for AUC computation
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {current_auc:.4f}"
        )
        fold_train_losses.append(avg_train_loss)
        fold_val_losses.append(avg_val_loss)
        fold_val_aucs.append(current_auc)
        if current_auc > best_auc:
            best_auc = current_auc
            save_path = SAVE_DIR / f"best_moe_fold{TRAIN_FOLD}.pth"
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

    onnx_save_path = os.path.join(str(SAVE_DIR), f"bird_moe_fold{TRAIN_FOLD}.onnx")
    plot_and_save_learning_curves(fold_train_losses, fold_val_losses, fold_val_aucs, TRAIN_FOLD, str(SAVE_DIR))
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    dummy_input = torch.randn(1, 1536).to(device)
    with open(os.devnull, "w") as f, redirect_stdout(f), redirect_stderr(f), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name in ("onnx", "onnxruntime", "torch.onnx", "onnxscript"):
            logging.getLogger(name).setLevel(logging.CRITICAL)
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            export_params=True,
            opset_version=15,
            do_constant_folding=True,
            input_names=["embedding"],
            output_names=["logits"],
            dynamic_axes={"embedding": {0: "batch_size"}, "logits": {0: "batch_size"}},
        )
    print(f"Exported ONNX model to {onnx_save_path}")
    baseline_fold_scores.append(best_auc)

print(f"CV score: {np.mean(baseline_fold_scores):.4f} (+/- {np.std(baseline_fold_scores):.4f})")


100%|██████████| 7110/7110 [00:02<00:00, 2486.62it/s]


TRAINING FOLD 0
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.2007 | Val Loss: 0.8381 | Val AUC: 0.9908
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold0.pth
Epoch 02/10 | Train Loss: 0.4874 | Val Loss: 0.8266 | Val AUC: 0.9905
Epoch 03/10 | Train Loss: 0.2842 | Val Loss: 0.8745 | Val AUC: 0.9903
Epoch 04/10 | Train Loss: 0.1967 | Val Loss: 0.9122 | Val AUC: 0.9876
Epoch 05/10 | Train Loss: 0.1553 | Val Loss: 0.9373 | Val AUC: 0.9884
Epoch 06/10 | Train Loss: 0.1423 | Val Loss: 0.9535 | Val AUC: 0.9883
Epoch 07/10 | Train Loss: 0.1436 | Val Loss: 0.9473 | Val AUC: 0.9892
Epoch 08/10 | Train Loss: 0.1350 | Val Loss: 0.9654 | Val AUC: 0.9870
Epoch 09/10 | Train Loss: 0.1235 | Val Loss: 0.9735 | Val AUC: 0.9863
Epoch 10/10 | Train Loss: 0.1288 | Val Loss: 1.0213 | Val AUC: 0.9847
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/learning_curves

100%|██████████| 7110/7110 [00:02<00:00, 3032.28it/s]


TRAINING FOLD 1
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.6833 | Val Loss: 0.6094 | Val AUC: 0.9970
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold1.pth
Epoch 02/10 | Train Loss: 0.3193 | Val Loss: 0.6427 | Val AUC: 0.9962
Epoch 03/10 | Train Loss: 0.1936 | Val Loss: 0.6866 | Val AUC: 0.9946
Epoch 04/10 | Train Loss: 0.1480 | Val Loss: 0.7524 | Val AUC: 0.9935
Epoch 05/10 | Train Loss: 0.1449 | Val Loss: 0.7766 | Val AUC: 0.9920
Epoch 06/10 | Train Loss: 0.1515 | Val Loss: 0.8162 | Val AUC: 0.9933
Epoch 07/10 | Train Loss: 0.1324 | Val Loss: 0.8127 | Val AUC: 0.9927
Epoch 08/10 | Train Loss: 0.1222 | Val Loss: 0.8397 | Val AUC: 0.9916
Epoch 09/10 | Train Loss: 0.1278 | Val Loss: 0.8693 | Val AUC: 0.9911
Epoch 10/10 | Train Loss: 0.1266 | Val Loss: 0.8740 | Val AUC: 0.9912
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/learning_curves

100%|██████████| 7110/7110 [00:01<00:00, 4449.58it/s]


TRAINING FOLD 2
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.4629 | Val Loss: 0.3930 | Val AUC: 0.9993
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold2.pth
Epoch 02/10 | Train Loss: 0.2449 | Val Loss: 0.4161 | Val AUC: 0.9987
Epoch 03/10 | Train Loss: 0.1611 | Val Loss: 0.4859 | Val AUC: 0.9981
Epoch 04/10 | Train Loss: 0.1494 | Val Loss: 0.5357 | Val AUC: 0.9976
Epoch 05/10 | Train Loss: 0.1426 | Val Loss: 0.6037 | Val AUC: 0.9952
Epoch 06/10 | Train Loss: 0.1358 | Val Loss: 0.6583 | Val AUC: 0.9949
Epoch 07/10 | Train Loss: 0.1368 | Val Loss: 0.6814 | Val AUC: 0.9942
Epoch 08/10 | Train Loss: 0.1253 | Val Loss: 0.7112 | Val AUC: 0.9940
Epoch 09/10 | Train Loss: 0.1248 | Val Loss: 0.7461 | Val AUC: 0.9927
Epoch 10/10 | Train Loss: 0.1294 | Val Loss: 0.7433 | Val AUC: 0.9933
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/learning_curves

100%|██████████| 7110/7110 [00:02<00:00, 3122.76it/s]


TRAINING FOLD 3
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.3374 | Val Loss: 0.2614 | Val AUC: 0.9992
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold3.pth
Epoch 02/10 | Train Loss: 0.1869 | Val Loss: 0.3229 | Val AUC: 0.9995
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold3.pth
Epoch 03/10 | Train Loss: 0.1439 | Val Loss: 0.3902 | Val AUC: 0.9992
Epoch 04/10 | Train Loss: 0.1365 | Val Loss: 0.4868 | Val AUC: 0.9985
Epoch 05/10 | Train Loss: 0.1452 | Val Loss: 0.5470 | Val AUC: 0.9978
Epoch 06/10 | Train Loss: 0.1342 | Val Loss: 0.5753 | Val AUC: 0.9972
Epoch 07/10 | Train Loss: 0.1323 | Val Loss: 0.6016 | Val AUC: 0.9971
Epoch 08/10 | Train Loss: 0.1149 | Val Loss: 0.6567 | Val AUC: 0.9959
Epoch 09/10 | Train Loss: 0.1111 | Val Loss: 0.6864 | Val AUC: 0.9959
Epoch 10/10 | Train Loss: 0.1243 | Val Loss: 0.7325 | Val AUC: 0.9940
Learni

100%|██████████| 7109/7109 [00:01<00:00, 4206.82it/s]


TRAINING FOLD 4
Training on 28440 samples, validating on 7109 samples
Epoch 01/10 | Train Loss: 0.2469 | Val Loss: 0.1940 | Val AUC: 0.9996
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold4.pth
Epoch 02/10 | Train Loss: 0.1604 | Val Loss: 0.2382 | Val AUC: 0.9997
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold4.pth
Epoch 03/10 | Train Loss: 0.1301 | Val Loss: 0.3115 | Val AUC: 0.9995
Epoch 04/10 | Train Loss: 0.1292 | Val Loss: 0.4151 | Val AUC: 0.9984
Epoch 05/10 | Train Loss: 0.1389 | Val Loss: 0.4490 | Val AUC: 0.9986
Epoch 06/10 | Train Loss: 0.1205 | Val Loss: 0.5019 | Val AUC: 0.9970
Epoch 07/10 | Train Loss: 0.1270 | Val Loss: 0.5587 | Val AUC: 0.9971
Epoch 08/10 | Train Loss: 0.1206 | Val Loss: 0.5827 | Val AUC: 0.9964
Epoch 09/10 | Train Loss: 0.1119 | Val Loss: 0.6384 | Val AUC: 0.9957
Epoch 10/10 | Train Loss: 0.1118 | Val Loss: 0.6437 | Val AUC: 0.9958
Learni